In [ ]:
# ENGE707 Project Phase-I: Task 3 - Data Cleansing and Transformation
# Dataset: medical_insurance.csv
# Target: risk_score (continuous, 0-1)

import pandas as pd
import numpy as np


df = pd.read_csv("medical_insurance.csv", keep_default_na=False, na_values=[""])
 
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")

Loaded 100,000 rows x 54 columns


In [ ]:
# To detect duplicates
assert df["person_id"].is_unique, "Duplicate person_id found"
assert df.duplicated().sum() == 0, "Duplicate rows found"

# alcohol_freq had problems with "none" being a missing value
# Confirm that no missing values remain
missing = df.isna().sum()
print("\nColumns with missing values after fixing the None/NaN parsing issue:")
print(missing[missing > 0] if missing.sum() else "(none)")


Columns with missing values after fixing the None/NaN parsing issue:
(none)


In [6]:
# Checking for if chronic_count should equal the sum of the individual disease flags

disease_cols = [
    "hypertension", "diabetes", "asthma", "copd", "cardiovascular_disease",
    "cancer_history", "kidney_disease", "liver_disease", "arthritis",
    "mental_health",
]
mismatch = (df[disease_cols].sum(axis=1) != df["chronic_count"]).sum()
print(f"\nRows where chronic_count disagrees with disease-flag sum: {mismatch}")


Rows where chronic_count disagrees with disease-flag sum: 0


In [ ]:
# A small nunber of records show age 0-1. Keeping them because they can be listed as dependants on a policy, but flagging them to keep decision visible.

df["age_flag_infant"] = df["age"] <= 1
print(f"\nRecords flagged as infants (age<=1), kept but flagged: {df['age_flag_infant'].sum()}")


Records flagged as infants (age<=1), kept but flagged: 214


In [8]:
# Some BMI values are at 12.0, below any possible adult BMI, may be floor/clipping artifact from date generation, flagging for now.

BMI_LOW, BMI_HIGH = 13, 55  # plausible physiological range
df["bmi_flag_outlier"] = ~df["bmi"].between(BMI_LOW, BMI_HIGH)
print(f"BMI values outside [{BMI_LOW}, {BMI_HIGH}] flagged: {df['bmi_flag_outlier'].sum()}")

BMI values outside [13, 55] flagged: 250


In [9]:
# Excluding values that are a direct result of risk_score, since they are unnecessary for predicting the target value

LEAKAGE_COLS = [
    "is_high_risk", "annual_premium", "monthly_premium", "annual_medical_cost",
    "claims_count", "avg_claim_amount", "total_claims_paid",
]
print(f"\nColumns flagged as target-leakage risk (exclude from explanatory model): {LEAKAGE_COLS}")


Columns flagged as target-leakage risk (exclude from explanatory model): ['is_high_risk', 'annual_premium', 'monthly_premium', 'annual_medical_cost', 'claims_count', 'avg_claim_amount', 'total_claims_paid']


In [10]:
categorical_cols = [
    "sex", "region", "urban_rural", "education", "marital_status",
    "employment_status", "smoker", "alcohol_freq", "plan_type", "network_tier",
]
for col in categorical_cols:
    df[col] = df[col].str.strip()  # guard against stray whitespace

# Ordinal columns will be encoded since they have a meaningful order. To be done tranformation.

In [11]:
# Save Cleaned dataset

df.to_csv("medical_insurance_cleaned.csv", index=False)
print(f"\nSaved cleaned dataset: medical_insurance_cleaned.csv ({df.shape[0]:,} rows x {df.shape[1]} columns)")


Saved cleaned dataset: medical_insurance_cleaned.csv (100,000 rows x 56 columns)
